## Importacion de librerias

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Union de dataset

In [2]:
sales_df = pd.read_csv("../data/sales_train.csv")
items = pd.read_csv("../data/items.csv")

In [3]:
sales_df.info()
sales_df.shape

<class 'pandas.DataFrame'>
RangeIndex: 2935849 entries, 0 to 2935848
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   date            str    
 1   date_block_num  int64  
 2   shop_id         int64  
 3   item_id         int64  
 4   item_price      float64
 5   item_cnt_day    float64
dtypes: float64(2), int64(3), str(1)
memory usage: 134.4 MB


(2935849, 6)

In [4]:
items.info()
items.shape

<class 'pandas.DataFrame'>
RangeIndex: 22170 entries, 0 to 22169
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   item_name         22170 non-null  str  
 1   item_id           22170 non-null  int64
 2   item_category_id  22170 non-null  int64
dtypes: int64(2), str(1)
memory usage: 519.7 KB


(22170, 3)

In [5]:
#item_id tienen en comun esa columna
df = sales_df.merge(items, on="item_id",how= "left") 

## DataSet 

In [6]:
df.shape

(2935849, 8)

In [7]:
df.head(3)

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,item_name,item_category_id
0,02.01.2013,0,59,22154,999.0,1.0,ЯВЛЕНИЕ 2012 (BD),37
1,03.01.2013,0,25,2552,899.0,1.0,DEEP PURPLE The House Of Blue Light LP,58
2,05.01.2013,0,25,2552,899.0,-1.0,DEEP PURPLE The House Of Blue Light LP,58


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2935849 entries, 0 to 2935848
Data columns (total 8 columns):
 #   Column            Dtype  
---  ------            -----  
 0   date              str    
 1   date_block_num    int64  
 2   shop_id           int64  
 3   item_id           int64  
 4   item_price        float64
 5   item_cnt_day      float64
 6   item_name         str    
 7   item_category_id  int64  
dtypes: float64(2), int64(4), str(2)
memory usage: 179.2 MB


## optimizacion de memoria sobre atributos

In [28]:
for col in df.columns:
    if df[col].dtype in ['int64', 'int32', 'float64']:
        maxim = df[col].max() 
        if maxim.max() < 127:
            print(f"Columna '{col}': Máximo {maxim} -> Sugerencia: int8")
        elif maxim.max() < 32767:
            print(f"Columna '{col}': Máximo {maxim} -> Sugerencia: int16")
        else:
            print(f"Columna '{col}': Máximo {maxim} -> Sugerencia: int32")

Columna 'item_price': Máximo 307980 -> Sugerencia: int32


In [29]:
print(df.select_dtypes(include=[np.number]).max())

date_block_num          33
shop_id                 33
item_id              22169
item_price          307980
item_cnt_day          2169
item_category_id        83
dtype: int32


In [30]:
df['date_block_num'] = df['date_block_num'].astype('int8')
df['shop_id'] = df['date_block_num'].astype('int8')
df['item_id'] = df['item_id'].astype('int16')
df['item_price'] = df['item_price'].astype('int32')
df['item_cnt_day'] = df['item_cnt_day'].astype('int16')
df['item_category_id'] = df['item_category_id'].astype('int8')

In [31]:
df['date'] = pd.to_datetime(df['date'], format='%d.%m.%Y')

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2935849 entries, 0 to 2935848
Data columns (total 8 columns):
 #   Column            Dtype         
---  ------            -----         
 0   date              datetime64[us]
 1   date_block_num    int8          
 2   shop_id           int8          
 3   item_id           int16         
 4   item_price        int32         
 5   item_cnt_day      int16         
 6   item_name         str           
 7   item_category_id  int8          
dtypes: datetime64[us](1), int16(2), int32(1), int8(3), str(1)
memory usage: 75.6 MB


## Variables

Uno de los errores que tuve en la prueba en el set de features no exlui la clumna sales, por lo que el modelo termina usando el target como input. En este dataset actual, la columna de ventas se llama `item_ctn_day`(cantidad de items vendidos por dia). Esta es la columna que se debe esconder del modelo durante la fase de features 

In [33]:
df.columns

Index(['date', 'date_block_num', 'shop_id', 'item_id', 'item_price',
       'item_cnt_day', 'item_name', 'item_category_id'],
      dtype='str')

En Machine Learning, no todos los números de identificación (IDs) son iguales. Mezclarlos puede arruinar el modelo. Los dividimos en dos tipos:

* 🔴 **El ID Tóxico (Identificador Único):** * **Ejemplo:** `boleta_id` o `transaccion_id`.
* **Comportamiento:** Nunca se repite. Aparece una sola vez por fila.
* **El Peligro:** Si el modelo lo ve, en lugar de aprender las reglas del negocio, simplemente "memoriza" la fila exacta (Sobreajuste / Overfitting).
* **Acción:** Eliminar siempre.


* 🟢 **El ID Categórico (Etiqueta de Agrupación):** * **Ejemplo:** `shop_id` (ID de la tienda) o `item_category_id` (ID de la categoría).
* **Comportamiento:** Se repiten miles de veces a lo largo del dataset.
* **El Valor:** Son la mina de oro del algoritmo. Le dicen al modelo: *"Esta venta ocurrió en la Tienda 5, que siempre vende más los viernes"*. Permiten agrupar y encontrar patrones de comportamiento.
* **Acción:** Conservar siempre, pero **transformar su tipo de dato**.


> **⚠️ Advertencia Matemática:** Si dejamos el `shop_id` como un número normal (entero), el modelo intentará hacer sumas o multiplicaciones con él, creyendo que la Tienda "50" vale más que la Tienda "2". Para evitar esto, debemos decirle a Pandas que convierta estas columnas al tipo `category`. Así, el modelo entenderá que son solo "etiquetas" o nombres disfrazados de números.


In [34]:
y = df["item_cnt_day"]
list_features= [col for col in df.columns if col not in ["date","item_cnt_day","item_name"]]
X = df[list_features]

## Holdout de 60 dias

En el mundo del Machine Learning tradicional (como predecir si un correo es spam o no), se suele mezclar todas las filas al azar y sacar un 20% para probar. Pero en Series de Tiempo (Retail), hacer eso es un pecado mortal. Si se mezcla los datos al azar, estaría usando las ventas de diciembre para predecir las de noviembre. ¡Viajaría en el tiempo!

Por eso, el "Holdout" debe ser estrictamente cronológico: cortamos la "cola" final del dataset (los últimos 60 días) y se esconde.

In [35]:
date_max = df["date"].max()
date_min = df["date"].min()
print(f"la fecha minima es: {date_min} | la fecha maxima es {date_max}")

la fecha minima es: 2013-01-01 00:00:00 | la fecha maxima es 2015-10-31 00:00:00


In [36]:
df.head(2)

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,item_name,item_category_id
0,2013-01-02,0,0,22154,999,1,ЯВЛЕНИЕ 2012 (BD),37
1,2013-01-03,0,0,2552,899,1,DEEP PURPLE The House Of Blue Light LP,58


### Calcular la fecha corte
Se crea la variable `cutoff`. A la fecha_maxima, se debe restar exactamente 60 días usando la herramienta de Pandas para medir lapsos de tiempo: ``pd.Timedelta(days=60)``.

In [37]:
cutoff = date_max - pd.Timedelta(days=60)

Como debe quedar la linea de tiempo: 

**[Entrenamiento / El Pasado]:** Todo desde 2013 hasta el día del corte. (El modelo estudia esto).

**[Validación / El Futuro Simulado]:** Esos últimos 60 días ocultos. (El modelo intenta predecir esto sin haberlo visto, y luego tú comparas sus respuestas con la realidad que guardaste).

In [38]:
train = df['date'] <= cutoff
valid = df['date'] > cutoff

In [39]:
X_train = X[train]
y_train = y[train]
X_valid = X[valid]
y_valid = y[valid]

In [40]:
print(f"Filas para entrenar (X_train): {X_train.shape[0]}")
print(f"Filas para validar (X_val): {X_valid.shape[0]}")

Filas para entrenar (X_train): 2834012
Filas para validar (X_val): 101837


In [41]:
date_future = pd.date_range(start=date_max + pd.Timedelta(days=1),periods=120, freq="D")
print(f"Fecha desde que empieza el pronostico: {date_future.min()}")
print(f"Fecha que termina el dia 120: {date_future.max()}")


Fecha desde que empieza el pronostico: 2015-11-01 00:00:00
Fecha que termina el dia 120: 2016-02-28 00:00:00


## Producto Cartesiano

En Machine Learning, los modelos no pueden predecir sobre "el vacío". Si queremos que el modelo nos diga cuánto venderá la **Tienda A** del **Producto X** el día **1 de Diciembre**, tenemos que entregarle una fila que diga exactamente eso, pero con el espacio de la venta vacío para que él lo rellene.

El problema es que el futuro tiene muchas combinaciones. Para no olvidar ninguna, usamos el **Producto Cartesiano**.

#### La Analogía del Menú

Imagina un restaurante que ofrece:

* **3 Entradas:** Sopa, Ensalada, Empanada.
* **2 Platos Principales:** Pasta, Carne.

¿Cuántas combinaciones de cena existen? El Producto Cartesiano dice que son $3 \times 2 = 6$ combinaciones posibles.

#### Aplicado a nuestro Forecast de Retail

Para nuestra predicción de 120 días, multiplicamos tres "listas de ingredientes":

1. **Fechas:** Los 120 días que generamos (Noviembre a Febrero).
2. **Tiendas:** Todas las sucursales únicas que tenemos en el histórico.
3. **Productos:** Todos los SKUs únicos que queremos predecir.

**La lógica es:**
`Día 1` $\times$ `Tienda 1` $\times$ `Producto 1`
`Día 1` $\times$ `Tienda 1` $\times$ `Producto 2`
...y así sucesivamente hasta cubrir cada rincón del catálogo en cada tienda para cada día.

#### El desafío del Data Engineer: La Explosión de Datos

El producto cartesiano crece muy rápido. Si tienes:

* 120 días
* 50 tiendas
* 20,000 productos

La tabla resultante tendrá **120,000,000 de filas** ($120 \times 50 \times 20,000$).
Para esto se debe usar herramientas eficientes como el `MultiIndex` de Pandas para manejar estos millones de registros sin que la memoria RAM colapse.


In [42]:
shop_uniques  = df['shop_id'].unique() 
items_uniques = df['item_id'].unique()

idx = pd.MultiIndex.from_product([date_future,shop_uniques,items_uniques])

In [43]:
df_future = pd.DataFrame(index=idx).reset_index()

In [44]:
df_future.info()

<class 'pandas.DataFrame'>
RangeIndex: 88972560 entries, 0 to 88972559
Data columns (total 3 columns):
 #   Column   Dtype         
---  ------   -----         
 0   level_0  datetime64[us]
 1   level_1  int8          
 2   level_2  int16         
dtypes: datetime64[us](1), int16(1), int8(1)
memory usage: 933.4 MB


In [48]:
df_future.columns = ['date', 'shop_id', 'item_id']

In [49]:
df_future = df_future.merge(items[['item_id', 'item_category_id']], on='item_id', how='left')
df_future['item_category_id'] = df_future['item_category_id'].astype('int8')

In [46]:
#definir las columnas de identidad
cats = ["shop_id","item_id","item_category_id"]
# Se toma el eje x para las categorias que va a modificar el modelo
for col in cats:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype("category")
        X_valid[col] = X_valid[col].astype("category")
X_train.info()
X_valid.info()

<class 'pandas.DataFrame'>
Index: 2834012 entries, 0 to 2882317
Data columns (total 5 columns):
 #   Column            Dtype   
---  ------            -----   
 0   date_block_num    int8    
 1   shop_id           category
 2   item_id           category
 3   item_price        int32   
 4   item_category_id  category
dtypes: category(3), int32(1), int8(1)
memory usage: 46.0 MB
<class 'pandas.DataFrame'>
Index: 101837 entries, 2831747 to 2935848
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype   
---  ------            --------------   -----   
 0   date_block_num    101837 non-null  int8    
 1   shop_id           101837 non-null  category
 2   item_id           101837 non-null  category
 3   item_price        101837 non-null  int32   
 4   item_category_id  101837 non-null  category
dtypes: category(3), int32(1), int8(1)
memory usage: 1.7 MB


In [ ]:
for col in cats:
    if col in df_future.columns:
        df_future[col] = df_future[col].astype("category")